# NutriSmart — Knowledge Graph (Neo4j Desktop, Ontology-Aligned)

**Database:** Neo4j Desktop · `bolt://localhost:7687`  
**Schema version:** ontology-aligned (`HAS_INGREDIENT`, `BELONGS_TO_CUISINE`, `HAS_NUTRITION`, `Cuisine`, `NutritionalProfile`)

This notebook:
1. Runs `ingest_desktop.py` to populate the Neo4j Desktop instance with the full ontology schema
2. Verifies node/edge counts and schema labels
3. Executes RQ-aligned Cypher queries and exports results

**Prerequisites**
- Neo4j Desktop running with database `nutrismart` started
- Python driver installed: `pip install neo4j pandas scipy`


In [1]:
## Cell 1 — Run ingestion script (ingest_desktop.py)
import subprocess, sys, os

# NOTE: Adjust the path to your ingest_desktop.py script as needed
result = subprocess.run([sys.executable, "ingest_desktop.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)


Connecting to Neo4j Desktop at bolt://localhost:7687 ...
Connected to Neo4j Desktop.

Database cleared.
Constraints and indexes created.

Ingesting data...
  Loaded 9,997 rows via _load_recipe_batch
  Loaded 58 rows via _load_ingredient_batch
  Loaded 103,908 rows via _load_has_ingredient_batch

Applying inference rules...
  Inference R1 applied: 9,997 recipes updated with total_co2e and carbon_tier

Creating NutritionalProfile nodes...
  NutritionalProfile nodes created: 9,997

=== Graph verification ===
  Recipe nodes                        9,997
  Ingredient nodes                       58
  Cuisine nodes                           5
  NutritionalProfile nodes            9,997
  HAS_INGREDIENT edges               96,139
  BELONGS_TO_CUISINE edges            9,997
  HAS_NUTRITION edges                 9,997
  Recipes w/ carbon_tier              9,997
  Recipes w/ total_co2e               9,997
  HAS_INGREDIENT w/ co2e             96,139

  Node labels:         ['Cuisine', 'Ingredient',

In [2]:
## Cell 2 — Connect driver (reused across all query cells)
# Target: Neo4j Desktop — password set when creating the database in Desktop app
from neo4j import GraphDatabase
import pandas as pd

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "nutrismart12311")

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("Connected to Neo4j Desktop.")


Connected to Neo4j Desktop.


In [3]:
## Cell 3 — Verification: node and edge counts + schema labels

def run_query(cypher):
    with driver.session() as s:
        return s.run(cypher).data()

checks = {
    "Recipe nodes":             "MATCH (r:Recipe)               RETURN count(r) AS n",
    "Ingredient nodes":         "MATCH (i:Ingredient)           RETURN count(i) AS n",
    "Cuisine nodes":            "MATCH (c:Cuisine)              RETURN count(c) AS n",
    "NutritionalProfile nodes": "MATCH (np:NutritionalProfile)  RETURN count(np) AS n",
    "HAS_INGREDIENT edges":     "MATCH ()-[h:HAS_INGREDIENT]->()     RETURN count(h) AS n",
    "BELONGS_TO_CUISINE edges": "MATCH ()-[b:BELONGS_TO_CUISINE]->() RETURN count(b) AS n",
    "HAS_NUTRITION edges":      "MATCH ()-[n:HAS_NUTRITION]->()      RETURN count(n) AS n",
    "Recipes w/ carbon_tier":   "MATCH (r:Recipe) WHERE r.carbon_tier IS NOT NULL RETURN count(r) AS n",
    "Recipes w/ total_co2e":    "MATCH (r:Recipe) WHERE r.total_co2e  IS NOT NULL RETURN count(r) AS n",
    "HAS_INGREDIENT w/ co2e":   "MATCH ()-[h:HAS_INGREDIENT]->() WHERE h.co2e_kg IS NOT NULL RETURN count(h) AS n",
}

print("=== Graph statistics ===")
for label, q in checks.items():
    n = run_query(q)[0]["n"]
    print(f"  {label:<30} {n:>10,}")

labels = [r["label"] for r in run_query("CALL db.labels() YIELD label RETURN label ORDER BY label")]
rels   = [r["relationshipType"] for r in run_query("CALL db.relationshipTypes() YIELD relationshipType RETURN relationshipType ORDER BY relationshipType")]
print(f"\n  Node labels:        {labels}")
print(f"  Relationship types: {rels}")

expected_labels = {"Cuisine", "Ingredient", "NutritionalProfile", "Recipe"}
expected_rels   = {"BELONGS_TO_CUISINE", "HAS_INGREDIENT", "HAS_NUTRITION"}
missing_l = expected_labels - set(labels)
missing_r = expected_rels - set(rels)
if missing_l or missing_r:
    print(f"\n  WARNING — missing labels: {missing_l}, missing rels: {missing_r}")
else:
    print("\n  Graph matches ontology schema. ✓")


=== Graph statistics ===
  Recipe nodes                        9,997
  Ingredient nodes                       58
  Cuisine nodes                           5
  NutritionalProfile nodes            9,997
  HAS_INGREDIENT edges               96,139
  BELONGS_TO_CUISINE edges            9,997
  HAS_NUTRITION edges                 9,997
  Recipes w/ carbon_tier              9,997
  Recipes w/ total_co2e               9,997
  HAS_INGREDIENT w/ co2e             96,139

  Node labels:        ['Cuisine', 'Ingredient', 'NutritionalProfile', 'Recipe']
  Relationship types: ['BELONGS_TO_CUISINE', 'HAS_INGREDIENT', 'HAS_NUTRITION']

  Graph matches ontology schema. ✓


In [4]:
## Cell 4 — Sample: top 10 ingredient CO₂ contributions across the graph
sample_q = """
MATCH (r:Recipe)-[h:HAS_INGREDIENT]->(i:Ingredient)
WHERE h.co2e_kg IS NOT NULL
RETURN r.name               AS recipe,
       i.ingredient_name    AS ingredient,
       round(h.grams, 1)    AS grams,
       round(h.co2e_kg, 4)  AS co2e_kg
ORDER BY h.co2e_kg DESC
LIMIT 10
"""
sample_df = pd.DataFrame(run_query(sample_q))
print("Top 10 ingredient CO₂ contributions across the graph:")
display(sample_df)


Top 10 ingredient CO₂ contributions across the graph:


,recipe,ingredient,grams,co2e_kg
0,Shorshe Ilish,beef,999.0,59.7902
1,Indian Desserts Special 351,beef,999.0,59.7902
2,Hakka Noodles,beef,999.0,59.7902
3,Bangladeshi Desserts Special 15,beef,999.0,59.7902
4,Fusion Street Food Special 240,beef,999.0,59.7902
5,Nihari,beef,998.0,59.7303
6,Mantu,beef,998.0,59.7303
7,Chapli Kabab,beef,997.0,59.6705
8,Haleem,beef,996.0,59.6106
9,Nihari,beef,995.0,59.5508


## Schema Alignment — carbon_tier Distribution

Uses the `carbon_tier` derived property set by inference rule R1 during ingestion (`ingest_desktop.py`).  
Tiers: `low` (< 2 kg CO₂eq) · `medium` (2–8 kg) · `high` (> 8 kg)


In [13]:
## carbon_tier distribution across all recipes
tier_q = """
MATCH (r:Recipe)
WHERE r.carbon_tier IS NOT NULL
RETURN r.carbon_tier AS tier, count(r) AS recipe_count
ORDER BY recipe_count DESC
"""
tier_df = pd.DataFrame(run_query(tier_q))
print("carbon_tier distribution:")
display(tier_df)


carbon_tier distribution:


,tier,recipe_count
0,high,4969
1,medium,3472
2,low,1556


In [14]:
## Schema verification — confirm graph labels and relationship types match ontology
label_q = "CALL db.labels() YIELD label RETURN label ORDER BY label"
rel_q   = "CALL db.relationshipTypes() YIELD relationshipType RETURN relationshipType ORDER BY relationshipType"

labels = [r["label"]            for r in run_query(label_q)]
rels   = [r["relationshipType"] for r in run_query(rel_q)]

print("=== Node labels in graph ===")
for lbl in labels:
    print(f"  {lbl}")

print("\n=== Relationship types in graph ===")
for rel in rels:
    print(f"  {rel}")

print("\n=== Expected ontology schema ===")
expected_labels = ["Cuisine", "Ingredient", "NutritionalProfile", "Recipe"]
expected_rels   = ["BELONGS_TO_CUISINE", "HAS_INGREDIENT", "HAS_NUTRITION"]

missing_labels = [l for l in expected_labels if l not in labels]
missing_rels   = [r for r in expected_rels   if r not in rels]

if missing_labels:
    for l in missing_labels:
        print(f"  MISSING label:        {l}")
if missing_rels:
    for r in missing_rels:
        print(f"  MISSING relationship: {r}")

if not missing_labels and not missing_rels:
    print("  Graph matches full ontology schema. ✓")


=== Node labels in graph ===
  Cuisine
  Ingredient
  NutritionalProfile
  Recipe

=== Relationship types in graph ===
  BELONGS_TO_CUISINE
  HAS_INGREDIENT
  HAS_NUTRITION

=== Expected ontology schema ===
  Graph matches full ontology schema. ✓


In [15]:
## Close driver
driver.close()
print("Driver closed.")


Driver closed.
